# 04 — Results Summary

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Mahekd/interpretable-nlp-sexism-detection.git
%cd interpretable-nlp-sexism-detection

import glob
import os
import shutil

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/EDOS_DATA/outputs_backup"

os.makedirs("outputs", exist_ok=True)
restored = 0
for src_path in glob.glob(os.path.join(DRIVE_BACKUP_DIR, "best_model_task*", "results.json")):
    run_dir = os.path.basename(os.path.dirname(src_path))
    dst_dir = os.path.join("outputs", run_dir)
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(src_path, os.path.join(dst_dir, "results.json"))
    restored += 1

print(f"Restored {restored} results.json file(s) from {DRIVE_BACKUP_DIR}")
if restored == 0:
    print("Nothing found -- check DRIVE_BACKUP_DIR, or that 02_training.ipynb actually backed up to Drive.")

Mounted at /content/drive
Cloning into 'interpretable-nlp-sexism-detection'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 134 (delta 67), reused 88 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 7.29 MiB | 13.70 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/interpretable-nlp-sexism-detection
Restored 34 results.json file(s) from /content/drive/MyDrive/EDOS_DATA/outputs_backup


In [5]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob('outputs/best_model_task*/results.json'):
    with open(path) as f:
        r = json.load(f)
    rows.append({
        'task': r['task'],
        'model': r['model_name'],
        'run_name': r.get('run_name', 'default'),
        'augment': r['augment'],
        'lr': r.get('lr'),
        'batch_size': r.get('batch_size'),
        'epochs_run': r.get('epochs_run'),
        'dev_macro_f1': round(r['dev_macro_f1'], 4),
        'test_macro_f1': round(r['test_macro_f1'], 4),
    })

results_df = pd.DataFrame(rows).sort_values(['task', 'model', 'dev_macro_f1'], ascending=[True, True, False]).reset_index(drop=True)
results_df

,task,model,run_name,augment,lr,batch_size,epochs_run,dev_macro_f1,test_macro_f1
0,A,bert-base-uncased,lr2e-5_bs16,none,0.00002,16,8,0.8271,0.8195
1,A,bert-base-uncased,lr3e-5_bs32,none,0.00003,32,6,0.8250,0.8175
2,A,bert-base-uncased,lr2e-5_bs32,none,0.00002,32,5,0.8243,0.8198
3,A,bert-base-uncased,lr3e-5_bs16,none,0.00003,16,10,0.8165,0.8268
4,A,bert-base-uncased,default,none,0.00002,16,4,0.8134,0.8217
5,A,roberta-base,default,none,0.00002,32,4,0.8324,0.8283
6,A,roberta-base,lr2e-5_bs32,none,0.00002,32,4,0.8320,0.8321
7,A,roberta-base,lr3e-5_bs16,none,0.00003,16,10,0.8281,0.8244
8,A,roberta-base,lr2e-5_bs16,none,0.00002,16,4,0.8258,0.8231
9,A,roberta-base,lr3e-5_bs32,none,0.00003,32,2,0.8232,0.8150


## Best run per task + model (by dev macro-F1)

In [ ]:
best_runs = (
    results_df
    .sort_values('dev_macro_f1', ascending=False)
    .groupby(['task', 'model'], as_index=False)
    .first()
    .sort_values(['task', 'model'])
    .reset_index(drop=True)
)
best_runs

,task,model,run_name,augment,lr,batch_size,epochs_run,dev_macro_f1,test_macro_f1
0,A,bert-base-uncased,lr2e-5_bs16,none,0.00002,16,8,0.8271,0.8195
1,A,roberta-base,default,none,0.00002,32,4,0.8324,0.8283
2,B,bert-base-uncased,default,none,0.00002,32,8,0.6453,0.5791
3,B,roberta-base,lr3e-5_bs16,none,0.00003,16,3,0.6567,0.5856
4,C,bert-base-uncased,default,none,0.00002,16,6,0.4732,0.4385
5,C,roberta-base,lr3e-5_bs16,none,0.00003,16,6,0.4482,0.4644


In [ ]:
pivot = results_df.pivot_table(index=['model', 'augment'], columns='task', values='test_macro_f1')
pivot

task                                   A        B        C
model             augment                                 
bert-base-uncased backtranslate      NaN  0.57765      NaN
                  none           0.82106  0.57798  0.41546
roberta-base      backtranslate      NaN  0.59820      NaN
                  none           0.82458  0.60002  0.44770

## Published EDOS baselines (proposal Table I, macro-F1)

In [ ]:
baselines = pd.DataFrame([
    {'system': 'Most Frequent Baseline', 'Task A': 0.431, 'Task B': 0.159, 'Task C': 0.032},
    {'system': 'DistilBERT Baseline',    'Task A': 0.780, 'Task B': 0.537, 'Task C': 0.314},
    {'system': 'DeBERTa-v3 Baseline',    'Task A': 0.824, 'Task B': 0.593, 'Task C': 0.317},
    {'system': 'Mahmoudi (BERT)',        'Task A': 0.830, 'Task B': 0.640, 'Task C': 0.470},
    {'system': 'Goldzycher (DeBERTa)',   'Task A': 0.859, 'Task B': 0.648, 'Task C': 0.449},
    {'system': 'Best SemEval System',    'Task A': 0.875, 'Task B': 0.720, 'Task C': 0.549},
]).set_index('system')
baselines

,Task A,Task B,Task C
system,,,
Most Frequent Baseline,0.431,0.159,0.032
DistilBERT Baseline,0.780,0.537,0.314
DeBERTa-v3 Baseline,0.824,0.593,0.317
Mahmoudi (BERT),0.830,0.640,0.470
Goldzycher (DeBERTa),0.859,0.648,0.449
Best SemEval System,0.875,0.720,0.549


## Faithfulness summary

In [ ]:
import os

FAITHFULNESS_SUMMARY_PATH = "/content/drive/MyDrive/EDOS_DATA/explainability_results/explainability_summary.csv"

if os.path.exists(FAITHFULNESS_SUMMARY_PATH):
    faithfulness_summary = pd.read_csv(FAITHFULNESS_SUMMARY_PATH)
else:
    print(f"Not found yet: {FAITHFULNESS_SUMMARY_PATH}")
    print("Run the persistence cell at the end of 03_explainability.ipynb first, "
          "then re-run this cell.")
    faithfulness_summary = pd.DataFrame()

faithfulness_summary

,task,model,mean_comprehensiveness,mean_sufficiency,lime_shap_agreement
0,A,BERT-base,0.249105,0.232418,0.380000
1,A,RoBERTa-base,0.224622,0.106265,0.500000
2,B,BERT-base,0.445061,0.252653,0.510000
3,B,RoBERTa-base,0.273121,0.066777,0.595000
4,C,BERT-base,0.366516,0.199625,0.508182
5,C,RoBERTa-base,0.411008,0.324837,0.463636


In [3]:
import json
import os

GLOBAL_SHAP_PATH = "/content/drive/MyDrive/EDOS_DATA/explainability_results/global_shap_importance.json"
BIAS_CONTROL_PATH = "/content/drive/MyDrive/EDOS_DATA/explainability_results/bias_control_comparison.json"


def _load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f"Not found yet: {path}")
    print("Run the persistence cell at the end of 03_explainability.ipynb first, "
          "then re-run this cell.")
    return {}


global_shap = _load_json(GLOBAL_SHAP_PATH)
bias_control = _load_json(BIAS_CONTROL_PATH)

In [6]:
top_tokens_rows = []
for key, result in global_shap.items():
    task, model_name = key.split("_", 1)
    for row in result["global_shap_importance"][:10]:
        top_tokens_rows.append({
            "task": task,
            "model": model_name,
            "token": row["token"],
            "n_occurrences": row["n_occurrences"],
            "mean_abs_shap": round(row["mean_abs_shap"], 4),
        })

global_shap_df = (
    pd.DataFrame(top_tokens_rows)
    .sort_values(["task", "model", "mean_abs_shap"], ascending=[True, True, False])
    .reset_index(drop=True)
)
global_shap_df

,task,model,token,n_occurrences,mean_abs_shap
0,A,bert-base-uncased,pussy,1,0.9981
1,A,bert-base-uncased,"bitch,",1,0.9981
2,A,bert-base-uncased,foids,1,0.9972
3,A,bert-base-uncased,dykes,1,0.8096
4,A,bert-base-uncased,raped,1,0.7062
5,A,bert-base-uncased,moms,1,0.6916
6,A,bert-base-uncased,woman,3,0.4807
7,A,bert-base-uncased,lady,1,0.3884
8,A,bert-base-uncased,tell,1,0.3742
9,A,bert-base-uncased,jews,1,0.3551


In [7]:
bias_rows = []
for key, result in bias_control.items():
    if result is None:
        continue
    task, model_name = key.split("_", 1)
    bias_rows.append({
        "task": task,
        "model": model_name,
        "gender_mean_abs_shap": round(result["gender_mean_abs_shap"], 4),
        "gender_n": result["gender_n"],
        "control_mean_abs_shap": round(result["control_mean_abs_shap"], 4),
        "control_n": result["control_n"],
        "observed_diff": round(result["observed_diff"], 4),
        "control_bootstrap_ci95": result["control_bootstrap_ci95"],
        "p_value_gender_not_higher": round(result["p_value_gender_not_higher"], 4),
    })

bias_control_df = (
    pd.DataFrame(bias_rows)
    .sort_values(["task", "model"])
    .reset_index(drop=True)
)
bias_control_df

,task,model,gender_mean_abs_shap,gender_n,control_mean_abs_shap,control_n,observed_diff,control_bootstrap_ci95,p_value_gender_not_higher
0,A,bert-base-uncased,0.1304,23,0.0384,206,0.0920,"[0.011721815562276637, 0.09385974287096017]",0.0020
1,A,roberta-base,0.1008,23,0.0419,206,0.0589,"[0.012986365066704702, 0.0938374974760066]",0.0165
2,B,bert-base-uncased,0.0978,69,0.0416,550,0.0562,"[0.029964731717410792, 0.055671042829460116]",0.0000
3,B,roberta-base,0.0790,69,0.0270,550,0.0519,"[0.018498218061741473, 0.03831925105414002]",0.0000
4,C,bert-base-uncased,0.0399,79,0.0219,806,0.0180,"[0.01783957895976032, 0.026593429808304297]",0.0000
5,C,roberta-base,0.0387,79,0.0213,806,0.0174,"[0.017505585517023244, 0.025324601358514182]",0.0000
